# Emoji provider classification: first submission

**Course:** Computer Vision (IDSI51006), BSc AIDAMS, ESSEC x CentraleSupélec
**Kaggle team:** _fill in_  |  **Group members:** _fill in_

This notebook starts from the example notebook sent with the project instructions and completes it:

1. **Loading fixed.** The images come in 6 different sizes and 4 colour modes, so `np.array(dataset)` in the example fails. Every image is flattened onto a white background and resized to 64 x 64 RGB.
2. **A real model in Step 4.** A small LeNet-style CNN (3 convolution + pooling blocks, then a dense layer), trained with Keras.
3. **Honest evaluation.** 15% of the training set is held out (stratified) to measure accuracy before submitting.
4. **Correct submission file.** `submission.csv` with columns `Id,Label`, where `Label` is the provider name (`apple`, `google`, ...), not a digit.

Measured in testing on a 2-core CPU: loading about 20 s, training about 5 min, validation accuracy about 81%. A Colab GPU runtime should be much faster (not measured).

## How to run

**Google Colab**
1. Download `2-computer-vision-2026-b-sc-aidams-final-proj.zip` from the Data tab of the Kaggle competition.
2. In Colab: *File > Upload notebook* and pick this file. Optional, faster: *Runtime > Change runtime type > T4 GPU*.
3. *Runtime > Run all*. In Step 2 an upload button appears: choose the zip (about 260 MB, so the upload can take a few minutes).
   To avoid re-uploading every session, put the zip once at the root of your Google Drive and set `USE_GOOGLE_DRIVE = True` in the configuration cell.
4. `submission.csv` downloads at the end. On Kaggle: *Submit Prediction* and upload it.

**Locally (VS Code or Jupyter)**
1. Put this notebook and the zip in the same folder.
2. Create an environment with **Python 3.11, 3.12 or 3.13** (TensorFlow has no wheels for 3.14 yet), for example:
   `python3.12 -m venv .venv` then `source .venv/bin/activate` (macOS/Linux) or `.venv\Scripts\activate` (Windows), then `pip install ipykernel`.
3. In VS Code, open the notebook, select the `.venv` kernel, then *Run All*. The first run installs the pinned versions listed in the setup cell.

## AI usage disclosure

This notebook was drafted with the help of Claude (Anthropic), then run and checked by the team. Prompt used:

> "look at this and make a suggestion for a first submissions off of an ipynb that I could run on colab or vs code locally" (with the project instructions email and the example notebook attached)

_Team: add any further prompts here as the project evolves. The report itself must be written by the team._

## STEP 1: Configuration, install and import libraries

In [ ]:
# ---- Configuration: the only cell you normally need to edit ----
SEED = 42                 # fixes the train/validation split and weight initialisation
IMG_SIZE = 64             # every image is resized to IMG_SIZE x IMG_SIZE RGB
VAL_FRACTION = 0.15       # share of the training set held out for validation
EPOCHS = 15               # maximum epochs; early stopping usually ends sooner
BATCH_SIZE = 64

# Use the original image size and colour mode as extra inputs (see Step 3 for why).
# False = the model only sees pixels (the baseline). True = image + size/mode, measured about +10 points.
USE_SIZE_AND_MODE = False

ZIP_NAME = "2-computer-vision-2026-b-sc-aidams-final-proj.zip"
USE_GOOGLE_DRIVE = False  # Colab only: look for the zip at the root of your Google Drive

# Tested versions, installed when running locally. Colab already ships every library used here.
LOCAL_REQUIREMENTS = [
    "tensorflow==2.21.0",
    "scikit-learn==1.8.0",
    "pandas==3.0.2",
    "numpy==2.4.4",
    "matplotlib==3.10.9",
    "pillow==12.2.0",
]

In [ ]:
import sys, subprocess, importlib.util

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    missing = [r.split("==")[0] for r in LOCAL_REQUIREMENTS
               if importlib.util.find_spec({"scikit-learn": "sklearn", "pillow": "PIL"}.get(r.split("==")[0], r.split("==")[0])) is None]
    if missing:
        print("Installing:", ", ".join(missing), "(one time only)")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *LOCAL_REQUIREMENTS])
        importlib.invalidate_caches()

print("Running on", "Google Colab" if IN_COLAB else "a local machine")

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # hide TensorFlow start-up noise

import platform, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import PIL
from PIL import Image
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(SEED)

VERSIONS = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "pillow": PIL.__version__,
}
for name, version in VERSIONS.items():
    print(f"{name:13s} {version}")
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU only)")

## STEP 2: Import and parse the data

The zip contains `train/` (9,879 PNG), `test/` (9,879 PNG) and `train_labels.csv` (`Id,Label`).
Two traps in the raw data, both handled below:

* **Mixed formats.** Images are 72x72, 240x240, 160x160 (plus a few others) and stored as RGBA, palette (`P`), grey + alpha (`LA`) or RGB. Stacking them with `np.array` fails, and converting RGBA straight to RGB can turn transparent backgrounds black or noisy. So each image is converted to RGBA, pasted onto a white background, converted to RGB and resized.
* **Id format.** File names are zero-padded (`00001.png`) while `train_labels.csv` stores `Id` as an integer (`1`). Labels are therefore joined on the integer Id, never on list order.

In [ ]:
WORK_DIR = Path("/content") if IN_COLAB else Path.cwd()

def find_data_dir():
    # Returns the folder containing train/, test/ and train_labels.csv, or None
    for base in (WORK_DIR / "data", WORK_DIR):
        if (base / "train").is_dir() and (base / "test").is_dir() and (base / "train_labels.csv").exists():
            return base
    return None

def find_zip():
    # Accepts renamed browser downloads such as '... (1).zip'
    search_dirs = [WORK_DIR]
    if IN_COLAB and USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        search_dirs.append(Path("/content/drive/MyDrive"))
    for folder in search_dirs:
        hits = sorted(folder.glob("*computer-vision*final-proj*.zip"))
        if hits:
            return hits[0]
    if IN_COLAB:
        from google.colab import files
        print(f"Please upload {ZIP_NAME}")
        uploaded = files.upload()
        return WORK_DIR / next(iter(uploaded))
    raise FileNotFoundError(f"Put {ZIP_NAME} in {WORK_DIR} (the folder of this notebook) and re-run.")

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    zip_path = find_zip()
    print("Extracting", zip_path.name, "...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK_DIR / "data")
    DATA_DIR = find_data_dir()
print("Data folder:", DATA_DIR)

### Function definition

In [ ]:
def load_image(path, size=IMG_SIZE):
    # Opens one emoji and returns (pixels, original width, original height, original colour mode).
    # Pixels: uint8 array of shape (size, size, 3), transparency flattened onto white.
    with Image.open(path) as img:
        width, height, mode = img.width, img.height, img.mode
        rgba = img.convert("RGBA")          # works for RGBA, P (palette), LA, L and RGB
    white = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
    rgb = Image.alpha_composite(white, rgba).convert("RGB")
    rgb = rgb.resize((size, size), Image.Resampling.BILINEAR)
    return np.asarray(rgb, dtype=np.uint8), width, height, mode


def load_folder(folder):
    # Loads every PNG of a folder, sorted by Id. Returns images, integer Ids and a table of original sizes/modes.
    files = sorted(Path(folder).glob("*.png"), key=lambda f: int(f.stem))
    images, info = [], []
    for f in files:
        pixels, width, height, mode = load_image(f)
        images.append(pixels)
        info.append({"Id": int(f.stem), "width": width, "height": height, "mode": mode})
    info = pd.DataFrame(info)
    return np.stack(images), info["Id"].to_numpy(), info


def show_img(imgs, titles, ncols=6, size=2.0):
    # Plots images in a grid with their titles
    nrows = int(np.ceil(len(imgs) / ncols))
    plt.figure(figsize=(ncols * size, nrows * size * 1.15))
    for i, (img, title) in enumerate(zip(imgs, titles)):
        plt.subplot(nrows, ncols, i + 1)
        plt.imshow(img)
        plt.title(title, fontsize=8)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## STEP 3: Data analysis
### Getting the shapes

In [ ]:
start = time.time()
X_train, train_ids, train_info = load_folder(DATA_DIR / "train")
X_test, test_ids, test_info = load_folder(DATA_DIR / "test")
print(f"Loaded in {time.time() - start:.0f} s")
print("Shape of train dataset", X_train.shape)
print("Shape of test dataset ", X_test.shape)

labels = pd.read_csv(DATA_DIR / "train_labels.csv")
labels["Id"] = labels["Id"].astype(int)
print("Shape of train labels ", labels.shape)

# Join labels to images on the integer Id
label_by_id = labels.set_index("Id")["Label"]
assert set(train_ids) == set(label_by_id.index), "Image files and train_labels.csv do not match"
y_train_text = label_by_id.loc[train_ids].astype(str).to_numpy()

CLASSES = sorted(labels["Label"].astype(str).unique())
class_to_index = {name: i for i, name in enumerate(CLASSES)}
y_train = np.array([class_to_index[name] for name in y_train_text])
print("Classes:", CLASSES)

### Class balance

Five providers have between 1,644 and 1,924 images each; `messenger` (580) and `mozilla` (397) are much rarer. That is why the macro-averaged F1 (every class counts equally) is reported next to accuracy.

In [ ]:
counts = pd.Series(y_train_text).value_counts()
print(counts.to_string())
counts.plot.bar(title="Training images per provider", rot=0, figsize=(8, 3))
plt.show()

### Showing example observations

In [ ]:
rng = np.random.default_rng(SEED)
imgs, titles = [], []
for name in CLASSES:
    picks = rng.choice(np.where(y_train_text == name)[0], size=6, replace=False)
    imgs += [X_train[i] for i in picks]
    titles += [f"{name} (Id {train_ids[i]})" for i in picks]
show_img(imgs, titles)

### Original size and colour mode per provider

Before resizing, the images carry a strong clue: their original size and colour mode depend on the provider. Every 160 x 160 image is WhatsApp (apart from 2 Apple), 240 x 240 images are Facebook, Messenger or Mozilla (apart from 3 Google), all 72 x 68 images are Google, and 91% of palette (`P`) images are Samsung.

This is a property of how the dataset was collected, not of how the emoji looks. Resizing to 64 x 64 erases it, so the baseline CNN has to recognise the drawing style alone. Setting `USE_SIZE_AND_MODE = True` gives these facts to the model as a second input. On the same validation split it moved accuracy from about 81% to about 91%. Whether that is a fair feature or a shortcut is a decision for the team to argue in the report.

In [ ]:
train_info["size"] = train_info["width"].astype(str) + "x" + train_info["height"].astype(str)
train_info["label"] = y_train_text
display(pd.crosstab(train_info["size"], train_info["label"]))
display(pd.crosstab(train_info["mode"], train_info["label"]))

In [ ]:
MODES = ["RGBA", "P", "LA", "RGB", "L"]

def size_mode_features(info):
    # Original width and height (divided by 240, the largest size) plus a one-hot colour mode
    columns = [info["width"] / 240, info["height"] / 240] + [(info["mode"] == m).astype(float) for m in MODES]
    return np.column_stack(columns).astype("float32")

M_train = size_mode_features(train_info)
M_test = size_mode_features(test_info)
print("Size/mode features:", M_train.shape)

## STEP 4: Train a model on the training dataset

**Validation split.** 15% of the training images are held out, stratified so every provider keeps its share. Kaggle gives only 5 submissions a day, so this split is how models are compared day to day.

**Model.** A LeNet-style CNN, deliberately small so it trains on a CPU:

* `Rescaling` maps pixels from 0-255 to 0-1.
* 3 blocks of `Conv2D` (3x3 filters: 32, then 64, then 128) + ReLU + `MaxPooling2D`. Each block halves the resolution (64 → 32 → 16 → 8) while learning more complex patterns: edges and colours first, then shading and outline styles.
* `Flatten`, `Dropout(0.4)` against overfitting, a `Dense(128)` layer, then a 7-way softmax.
* If `USE_SIZE_AND_MODE` is on, the size/mode vector goes through a small `Dense(16)` and joins the image features before the last dense layer.

**Training.** Adam optimiser, cross-entropy loss. Early stopping watches validation accuracy and keeps the best epoch.

In [ ]:
idx_fit, idx_val = train_test_split(
    np.arange(len(X_train)), test_size=VAL_FRACTION, stratify=y_train, random_state=SEED
)
print(f"Training on {len(idx_fit)} images, validating on {len(idx_val)}")

def model_inputs(X, M):
    return [X, M] if USE_SIZE_AND_MODE else X

def build_model():
    image_in = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1.0 / 255)(image_in)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.4)(x)
    inputs = [image_in]
    if USE_SIZE_AND_MODE:
        meta_in = keras.Input(shape=(M_train.shape[1],), name="size_and_mode")
        inputs.append(meta_in)
        x = layers.Concatenate()([x, layers.Dense(16, activation="relu")(meta_in)])
    x = layers.Dense(128, activation="relu")(x)
    output = layers.Dense(len(CLASSES), activation="softmax")(x)
    return keras.Model(inputs if USE_SIZE_AND_MODE else image_in, output, name="lenet_style_cnn")

model = build_model()
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
start = time.time()
history = model.fit(
    model_inputs(X_train[idx_fit], M_train[idx_fit]), y_train[idx_fit],
    validation_data=(model_inputs(X_train[idx_val], M_train[idx_val]), y_train[idx_val]),
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)],
)
print(f"Training time: {time.time() - start:.0f} s")

hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
for ax in axes:
    ax.set_xlabel("epoch")
plt.show()

### Validation results

A gap between training and validation accuracy means overfitting. The confusion matrix shows which providers get mixed up (Apple, Google and Samsung share the same 72 x 72 format, for example).

In [ ]:
val_proba = model.predict(model_inputs(X_train[idx_val], M_train[idx_val]), verbose=0)
val_pred = val_proba.argmax(axis=1)
val_true = y_train[idx_val]

print(f"Validation accuracy: {accuracy_score(val_true, val_pred):.3f}")
print(f"Validation macro F1: {f1_score(val_true, val_pred, average='macro'):.3f}\n")
print(classification_report(val_true, val_pred, target_names=CLASSES, digits=3))

ConfusionMatrixDisplay.from_predictions(
    val_true, val_pred, display_labels=CLASSES, xticks_rotation=45, cmap="Blues", colorbar=False
)
plt.title("Validation confusion matrix")
plt.show()

In [ ]:
# A few validation mistakes, useful to understand what the model finds hard
wrong = np.where(val_pred != val_true)[0][:12]
show_img(
    [X_train[idx_val[i]] for i in wrong],
    [f"true {CLASSES[val_true[i]]}\npred {CLASSES[val_pred[i]]}" for i in wrong],
)

## STEP 5: Make predictions and build the submission

Predictions are provider names in the same spelling as `train_labels.csv`. The file has one row per test image, `Id` as an integer.

In [ ]:
test_pred = model.predict(model_inputs(X_test, M_test), verbose=0).argmax(axis=1)
submission_df = pd.DataFrame({"Id": test_ids, "Label": [CLASSES[i] for i in test_pred]})

# Sanity checks before uploading to Kaggle
n_test_files = len(list((DATA_DIR / "test").glob("*.png")))
assert len(submission_df) == n_test_files, "One row per test image expected"
assert submission_df["Id"].is_unique
assert set(submission_df["Label"]) <= set(CLASSES)

SUBMISSION_PATH = WORK_DIR / "submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)
print("Saved", SUBMISSION_PATH, "with", len(submission_df), "rows")
print(submission_df.head())
print("\nPredicted label counts:\n" + submission_df["Label"].value_counts().to_string())

In [ ]:
# Record the exact library versions of this run (to lock them for the final codebase)
(WORK_DIR / "versions_used.txt").write_text("\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n")

if IN_COLAB:
    from google.colab import files
    files.download(str(SUBMISSION_PATH))

## Next steps (ideas for the experiment tracker)

Each idea changes one thing, so its effect can be measured on the same validation split before spending a Kaggle submission.

* **Size and mode input:** `USE_SIZE_AND_MODE = True` (about +10 points on validation in testing). Needs a clear justification in the report.
* **Class imbalance:** pass `class_weight` to `fit` so `messenger` and `mozilla` count more, then compare their recall.
* **Resolution:** `IMG_SIZE = 96`. The 240 x 240 images lose detail at 64.
* **Regularisation:** BatchNormalization and light augmentation (small shifts and zooms). Slower per epoch; worth a GPU runtime.
* **Transfer learning:** a pretrained network from `keras.applications` (e.g. MobileNetV2) with a new 7-class head. Pretrained models are allowed; use a GPU runtime.
* **Final fit:** once the settings are chosen, retrain on 100% of the training data for the same number of epochs before the final submission.

**Before the final submission, lock versions:** Colab's preinstalled libraries change over time. Copy `versions_used.txt` into the README and pin those versions (a `%pip install` cell at the top) so the notebook replays identically when it is graded.